# Step 5 Part C: Training loop

First, build the val tensors (using train's normalization stats -- never recompute stats on val/test). Then train the LSTM with mini-batch gradient descent on the CVaR loss, checking val performance periodically, and saving the best checkpoint.

## Build val tensors using TRAIN's normalization stats (reused from step 5a)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import json
from scipy.stats import norm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BTC_TRANSACTION_COST_RATE = 0.0005
MAX_LEN = 24
MIN_LEN = 4
feature_names = ["moneyness", "ttm", "iv", "delta", "is_call"]
norm_stats = json.load(open("norm_stats.json"))

def bs_delta(S, K, T, sigma, option_type, r=0.0):
    valid = (T > 0) & (sigma > 0)
    d1 = np.where(valid, (np.log(S / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(np.where(valid, T, 1))), 0)
    call_delta = norm.cdf(d1)
    put_delta = call_delta - 1.0
    is_call = (option_type == "call")
    return np.where(valid, np.where(is_call, call_delta, put_delta), 0.0)

def build_episode_arrays(df, max_len=MAX_LEN, min_len=MIN_LEN):
    features_list, spot_list, option_pnl_list, mask_list = [], [], [], []
    meta = []
    for (symbol, sample_date), group in df.groupby(["symbol", "sample_date"]):
        group = group.sort_values("hour_bucket").reset_index(drop=True)
        n = len(group)
        if n < min_len:
            continue
        n_use = min(n, max_len)
        group = group.iloc[:n_use]
        moneyness = (group["underlying_price"] / group["strike_price"]).values
        ttm = group["T_years"].values
        iv = group["iv_decimal"].clip(upper=3.0).values
        delta = group["bs_delta"].values
        is_call = (group["type"] == "call").astype(float).values
        feat = np.stack([moneyness, ttm, iv, delta, is_call], axis=1)
        spot = group["underlying_price"].values
        option_mid_usd = group["option_mid_usd"].values
        option_pnl = -np.diff(option_mid_usd, prepend=option_mid_usd[0])
        option_pnl[0] = 0.0
        pad_n = max_len - n_use
        if pad_n > 0:
            feat = np.vstack([feat, np.zeros((pad_n, feat.shape[1]))])
            spot = np.concatenate([spot, np.full(pad_n, spot[-1])])
            option_pnl = np.concatenate([option_pnl, np.zeros(pad_n)])
        mask = np.array([1.0] * n_use + [0.0] * pad_n)
        features_list.append(feat)
        spot_list.append(spot)
        option_pnl_list.append(option_pnl)
        mask_list.append(mask)
        meta.append({"symbol": symbol, "sample_date": sample_date, "n_steps": n_use})
    return (np.stack(features_list), np.stack(spot_list), np.stack(option_pnl_list),
            np.stack(mask_list), pd.DataFrame(meta))

def normalize_features(features, stats):
    normed = features.copy()
    for i, name in enumerate(feature_names):
        normed[:, :, i] = (features[:, :, i] - stats[name]["mean"]) / stats[name]["std"]
    return normed

val = pd.read_csv("btc_options_val.csv")
val["hour_bucket"] = pd.to_datetime(val["hour_bucket"])
val["sample_date"] = val["hour_bucket"].dt.date
val["T_years"] = val["time_to_maturity_days"] / 365
val["iv_decimal"] = val["mark_iv"] / 100
val["option_mid_usd"] = val["mid_price"] * val["underlying_price"]
val["bs_delta"] = bs_delta(val["underlying_price"].values, val["strike_price"].values,
                              val["T_years"].values, val["iv_decimal"].values, val["type"].values)

val_features, val_spots, val_option_pnls, val_masks, val_meta = build_episode_arrays(val)
val_features_normed = normalize_features(val_features, norm_stats)
print(f"Val: {val_features.shape[0]} episodes")

np.savez("val_episode_tensors.npz", features=val_features_normed, spots=val_spots,
         option_pnls=val_option_pnls, masks=val_masks)
val_meta.to_csv("val_episode_meta.csv", index=False)

## Load everything and set up the model, optimizer, data loaders

In [ ]:
train_data = np.load("train_episode_tensors.npz")
train_features = torch.tensor(train_data["features"], dtype=torch.float32)
train_spots = torch.tensor(train_data["spots"], dtype=torch.float32)
train_option_pnls = torch.tensor(train_data["option_pnls"], dtype=torch.float32)
train_masks = torch.tensor(train_data["masks"], dtype=torch.float32)

val_features_t = torch.tensor(val_features_normed, dtype=torch.float32).to(device)
val_spots_t = torch.tensor(val_spots, dtype=torch.float32).to(device)
val_option_pnls_t = torch.tensor(val_option_pnls, dtype=torch.float32).to(device)
val_masks_t = torch.tensor(val_masks, dtype=torch.float32).to(device)

class DeepHedgePolicy(nn.Module):
    def __init__(self, input_size=5, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        raw_position = self.head(lstm_out)
        return 1.5 * torch.tanh(raw_position.squeeze(-1))

def simulate_pnl_batch(positions, spots, option_pnls, masks, cost_rate=BTC_TRANSACTION_COST_RATE):
    batch_size, seq_len = positions.shape
    prev_position = torch.cat([torch.zeros(batch_size, 1, device=positions.device), positions[:, :-1]], dim=1)
    trade = (positions - prev_position) * masks
    cost = trade.abs() * spots * (cost_rate / 2)
    prev_spot = torch.cat([spots[:, :1], spots[:, :-1]], dim=1)
    hedge_pnl = prev_position * (spots - prev_spot)
    total_pnl_per_step = (option_pnls + hedge_pnl - cost) * masks
    return total_pnl_per_step.sum(dim=1), total_pnl_per_step

def cvar_loss(terminal_pnl, alpha=0.95):
    losses = -terminal_pnl
    k = max(1, int((1 - alpha) * losses.shape[0]))
    worst_losses, _ = torch.topk(losses, k)
    return worst_losses.mean()

model = DeepHedgePolicy().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(f"Train episodes: {train_features.shape[0]}, Val episodes: {val_features_t.shape[0]}")

## Training loop

In [ ]:
N_EPOCHS = 50
BATCH_SIZE = 256
n_train = train_features.shape[0]

train_losses, val_losses = [], []
best_val_loss = float("inf")
best_state = None

for epoch in range(N_EPOCHS):
    model.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    n_batches = 0

    for start in range(0, n_train, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        batch_features = train_features[idx].to(device)
        batch_spots = train_spots[idx].to(device)
        batch_option_pnls = train_option_pnls[idx].to(device)
        batch_masks = train_masks[idx].to(device)

        optimizer.zero_grad()
        positions = model(batch_features)
        terminal_pnl, _ = simulate_pnl_batch(positions, batch_spots, batch_option_pnls, batch_masks)
        loss = cvar_loss(terminal_pnl)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)

    model.eval()
    with torch.no_grad():
        val_positions = model(val_features_t)
        val_terminal_pnl, _ = simulate_pnl_batch(val_positions, val_spots_t, val_option_pnls_t, val_masks_t)
        val_loss = cvar_loss(val_terminal_pnl).item()
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch == N_EPOCHS - 1:
        print(f"Epoch {epoch:3d} | train CVaR loss: {avg_train_loss:10.4f} | val CVaR loss: {val_loss:10.4f}")

print(f"\nBest val CVaR loss: {best_val_loss:.4f}")
torch.save(best_state, "best_deep_hedge_model.pt")
print("Saved best_deep_hedge_model.pt")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label="Train CVaR loss")
ax.plot(val_losses, label="Val CVaR loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("CVaR loss (lower is better)")
ax.legend()
ax.set_title("Training curves")
plt.tight_layout()
plt.show()